In [ ]:
!pip install -q scikit-learn seaborn --upgrade
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, shutil
from pathlib import Path

DRIVE_DATASET_PATH = "/content/drive/MyDrive/Nepali_vehicle_dataset"

LOCAL_RAW_DIR = Path("/content/data/raw")
LOCAL_RAW_DIR.mkdir(parents=True, exist_ok=True)



print("\n---- Top-level structure under /content/data/raw ----")
for p in sorted(LOCAL_RAW_DIR.rglob("*")):
    if p.is_dir() and len(p.relative_to(LOCAL_RAW_DIR).parts) <= 2:
        print(p)


In [ ]:
RAW_DIR = "/content//drive/MyDrive/Nepali_vehicle_dataset"
assert os.path.isdir(RAW_DIR), f"{RAW_DIR} does not exist — fix RAW_DIR based on the printed tree above."
class_folders = sorted([d for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d))])
print(f"Found {len(class_folders)} class folders:")
print(class_folders)


In [ ]:
import json, random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import img_to_array, load_img

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, f1_score,
)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
set_seed()

IMG_SIZE = 224
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

SPLIT_DIR = Path("/content/data/splits")
RESULTS_DIR = Path("/content/results"); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("/content/models"); MODELS_DIR.mkdir(parents=True, exist_ok=True)
